# Stage 6B: deterministic sensor adaptation

This notebook applies normalized sensor observations to an existing one-day plan. Policy values are configurable, non-medical prototype product parameters pending user testing and hardware-team confirmation.

In [ ]:
import json
from pathlib import Path

from src.sensor_adapter import SensorPlanAdapter

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
plans = {item['example_id']: item['plan'] for item in json.loads((project_root / 'data' / 'processed' / 'planner_demo_outputs.json').read_text(encoding='utf-8'))}
observations = {item['example_id']: item['observation'] for item in json.loads((project_root / 'data' / 'processed' / 'sensor_demo_inputs.json').read_text(encoding='utf-8'))}
adapter = SensorPlanAdapter(project_root)
original_plan = plans['school_evening']
print(original_plan['plan_id'], original_plan['total_planned_minutes'])

## Normal and triggered observations

A normal observation keeps the timer plan. Long sitting changes only future session timing.

In [ ]:
normal = adapter.adapt(original_plan, observations['normal_observation'])
long_original = plans['short_window']
long_sitting = adapter.adapt(long_original, observations['long_sitting'])
print({'normal_applied': normal.adaptation_applied, 'long_actions': [item.action for item in long_sitting.actions]})
assert not normal.adaptation_applied
assert 'long_continuous_sitting' in long_sitting.triggers

## Posture and missing-data behavior

A sustained directional observation and pressure imbalance share one repositioning break. Missing data calmly restores the original timer plan.

In [ ]:
posture = adapter.adapt(original_plan, observations['posture_and_pressure'])
missing = adapter.adapt(original_plan, observations['missing_sensor_fallback'])
print({'posture_triggers': list(posture.triggers), 'posture_notice': posture.sensor_notice})
print({'missing_mode': missing.mode, 'missing_notice': missing.sensor_notice})
assert sum(item['session_type'] == 'break' for item in posture.sessions) == sum(item['session_type'] == 'break' for item in original_plan['sessions'])
assert list(missing.sessions) == original_plan['sessions']

## Before-and-after timeline

Only type, duration, and academic subject are shown here so timing changes are easy to inspect.

In [ ]:
def timeline(sessions):
    return [(item['order'], item['session_type'], item['duration_minutes'], item['subject']) for item in sessions]

print('Before:', timeline(long_original['sessions']))
print('After: ', timeline(long_sitting.sessions))

## Time and determinism checks

Every adapted result stays inside the original available time, and repeated input produces the same full output and ID.

In [ ]:
results = [normal, long_sitting, posture, missing]
for result in results:
    assert result.total_planned_minutes <= result.total_available_minutes
    assert all(item['duration_minutes'] >= 20 for item in result.sessions if item['session_type'] == 'study')
repeat = adapter.adapt(long_original, observations['long_sitting'])
assert repeat.to_dict() == long_sitting.to_dict()
print({'time_checks': 'passed', 'deterministic_id': repeat.adapted_plan_id})